# byteseek in Python

`byteseek` compiles literal patterns once and searches strings, files, and directories in Rust, releasing the GIL while it searches. It also parses JSON, YAML, and TOML and searches them by [jq](https://jqlang.org)-style path, by value, or with predicates.


In [1]:
import json
import pickle
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

from byteseek import (
    CaseMode,
    Match,
    PathOrder,
    SearchConfig,
    Searcher,
    StructuredMatch,
    StructuredMode,
    StructuredPredicateHandling,
)

## Searching text

A `Searcher` compiles its patterns, a string or an iterable of strings, once and is reused for every search. Each `Match` has `start` and `end` offsets and the `pattern_index` of the pattern that matched.


In [ ]:
searcher = Searcher(["error", "timeout"])
log = "worker-1 error: upstream timeout\nworker-2 ok\nworker-3 error: retrying"

print(searcher)
print(searcher.find(log))
print("first:", searcher.find_first(log))
print(
    "count:", searcher.count(log), "contains 'all good':", searcher.contains("all good")
)

Searcher(['error', 'timeout'])
[Match(start=9, end=14, pattern_index=0), Match(start=25, end=32, pattern_index=1), Match(start=54, end=59, pattern_index=0)]
first: Match(start=9, end=14, pattern_index=0)
count: 3 contains 'all good': False


In [3]:
for found in searcher.find(log):
    match found:
        case Match(start, end, pattern_index=1):
            print(f"timeout at {start}..{end}")
        case Match(start, end):
            print(f"error at {start}..{end}")

print("equal across searches:", searcher.find(log) == searcher.find(log))

error at 9..14
timeout at 25..32
error at 54..59
equal across searches: True


> **Warning:** `start` and `end` are UTF-8 **byte** offsets, not `str` indices. They agree only while the text before a match is ASCII. Slice the encoded text instead:


In [ ]:
menu = "café crème, café noir"
for found in Searcher("café").find(menu):
    by_bytes = menu.encode()[found.start : found.end].decode()
    by_index = menu[found.start : found.end]
    print(
        f"bytes {found.start}-{found.end}: {by_bytes!r}, but str slicing gives {by_index!r}"
    )

bytes 0-5: 'café', but str slicing gives 'café '
bytes 14-19: 'café', but str slicing gives 'fé no'


## Configuring a search

`SearchConfig` takes keyword arguments for case folding, whole words, overlapping matches, match limits, fuzzy matching, boolean queries, structured formats, and file traversal. Its `repr` lists the options that differ from the defaults. Enum options also accept their string values, such as `case_mode="ascii_insensitive"`.


In [ ]:
sample = "WARN warning Warn warn_level warn"
words = Searcher(
    "warn", SearchConfig(case_mode=CaseMode.ASCII_INSENSITIVE, whole_word=True)
)
print(words)
print([sample[found.start : found.end] for found in words.find(sample)])

limited = Searcher("warn", SearchConfig(case_mode="ascii_insensitive", max_matches=2))
print("max_matches=2:", len(limited.find(sample)), "matches")

overlapping = Searcher("aba", SearchConfig(overlapping=True))
print(
    "overlapping:",
    len(overlapping.find("ababa")),
    "vs",
    len(Searcher("aba").find("ababa")),
)

Searcher(['warn'], SearchConfig(case_mode=CaseMode.ASCII_INSENSITIVE, whole_word=True))
['WARN', 'Warn', 'warn']
max_matches=2: 2 matches
overlapping: 2 vs 1


## Lines, fuzzy matches, and boolean queries

`find_lines` returns each matching line with its 1-based number; offsets in `LineMatch.matches` are byte offsets into `line`. `find_lines(text, parallel=True)` returns the same lines, searching large texts on several threads.

`max_edit_distance` also matches text within that many insertions, deletions, or substitutions of a pattern, which catches typos such as `timout`. With `boolean_queries`, each pattern is a query that selects lines containing every required term, in any order.


In [6]:
service_log = "\n".join(
    [
        "INFO  payments status=200 latency_ms=40",
        "ERROR payments status=500 upstream timeout",
        "WARN  search status=200 timout budget nearly exhausted",
        "ERROR payments status=502 bad gateway",
    ]
)
line_searches = {
    "ERROR or timeout": Searcher(["ERROR", "timeout"]),
    "timeout, within one edit": Searcher("timeout", SearchConfig(max_edit_distance=1)),
    "ERROR AND payments AND NOT 502": Searcher(
        "ERROR AND payments AND NOT 502", SearchConfig(boolean_queries=True)
    ),
}
for label, line_searcher in line_searches.items():
    print(label)
    for line in line_searcher.find_lines(service_log):
        print(f"  {line.line_number}: {line.line}")

ERROR or timeout
  2: ERROR payments status=500 upstream timeout
  4: ERROR payments status=502 bad gateway
timeout, within one edit
  2: ERROR payments status=500 upstream timeout
  3: WARN  search status=200 timout budget nearly exhausted
ERROR AND payments AND NOT 502
  2: ERROR payments status=500 upstream timeout


## Files and directories

`find_path` searches a file, or every file under a directory, skipping hidden and binary files and files a `.gitignore` excludes. Results come in discovery order unless `path_order=PathOrder.PATH` asks for path order.


In [ ]:
workspace = TemporaryDirectory()
root = Path(workspace.name)
(root / ".git").mkdir()  # .gitignore files apply inside repositories
(root / "src").mkdir()
(root / ".gitignore").write_text("ignored.log\n")
(root / "app.log").write_text("INFO ready\nERROR failed\n")
(root / "ignored.log").write_text("ERROR ignored\n")
(root / "src" / "lib.rs").write_text("// ERROR in code\n")


def found_files(config: SearchConfig) -> list[tuple[str, int]]:
    """Searches the workspace for ERROR and returns each file's relative path and match count."""
    results = Searcher("ERROR", config).find_path(root)
    return [
        (Path(result.path).relative_to(root).as_posix(), len(result.matches))
        for result in results
    ]


print(found_files(SearchConfig(path_order=PathOrder.PATH)))
print(found_files(SearchConfig(path_order=PathOrder.PATH, ignore_gitignore=True)))

[('app.log', 1), ('src/lib.rs', 1)]
[('app.log', 1), ('ignored.log', 1), ('src/lib.rs', 1)]


## Structured search

With `structured_mode`, each text or file is parsed as a JSON, YAML, or TOML document, and every pattern matches one of three ways:

- Plain text matches inside scalar values, such as `failed`.
- A pattern starting with `$` matches a path exactly, such as `$.events[2].service`, or a path with its value, such as `$.service=payments`.
- A predicate compares values with `==`, `!=`, `>`, `>=`, `<`, `<=`, or `BETWEEN`, and its clauses join with `AND`. Its path matches at any depth, and all clauses must hold within the same object.

Each `StructuredMatch` has the jq-style `path` of the matched value, its `node_text`, its byte offsets, and a `match_kind`.


In [ ]:
def show(matches: list[StructuredMatch]) -> None:
    """Prints each structured match's pattern index, kind, path, and text."""
    width = max((len(found.path) for found in matches), default=0)
    for found in matches:
        print(
            f"{found.pattern_index}  {found.match_kind:10}  {found.path:{width}}  {found.node_text}"
        )


events_json = json.dumps(
    {
        "events": [
            {"service": "payments", "payload": {"status": "ok", "latency_ms": 180}},
            {
                "service": "payments",
                "payload": {"status": "failed", "latency_ms": 3011},
            },
            {"service": "search", "payload": {"status": "ok", "latency_ms": 1400}},
            {"service": "orders", "payload": {"status": "failed", "latency_ms": 95}},
        ]
    },
    indent=2,
)
events = Searcher(
    ["failed", "$.payload.latency_ms > 1000", "$.events[2].service"],
    SearchConfig(structured_mode=StructuredMode.JSON),
)
show(events.find_structured(events_json))

1  predicate   $.events[1].payload.latency_ms  3011
0  value       $.events[1].payload.status      "failed"
1  predicate   $.events[2].payload.latency_ms  1400
2  path        $.events[2].service             "search"
0  value       $.events[3].payload.status      "failed"


A predicate with several clauses reports one match per object that satisfies them all, at its first clause's value:


In [9]:
slow_failures = Searcher(
    '$.payload.status == "failed" AND $.payload.latency_ms > 1000',
    SearchConfig(structured_mode=StructuredMode.JSON),
)
show(slow_failures.find_structured(events_json))

0  predicate   $.events[1].payload.status  "failed"


YAML and TOML documents are searched the same way:


In [10]:
deployment_yaml = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
spec:
  replicas: 3
  template:
    spec:
      containers:
        - name: web
          image: web:1.4
          ports:
            - containerPort: 8080
        - name: sidecar
          image: envoy:1.31
"""
cargo_toml = """\
[package]
name = "service"
edition = "2021"
rust-version = "1.85"

[dependencies]
tokio = { version = "1.48", features = ["full"] }
serde = "1.0"
"""
deployment = Searcher(
    ["$.spec.replicas=3", "envoy", "$.containerPort >= 8000"],
    SearchConfig(structured_mode=StructuredMode.YAML),
)
manifest = Searcher(
    ['$.package["rust-version"]', "$.dependencies.tokio.version", "1.0"],
    SearchConfig(structured_mode=StructuredMode.TOML),
)
show(deployment.find_structured(deployment_yaml))
print()
show(manifest.find_structured(cargo_toml))

0  path_value  $.spec.replicas                                            3
2  predicate   $.spec.template.spec.containers[0].ports[0].containerPort  8080
1  value       $.spec.template.spec.containers[1].image                   envoy:1.31

0  path        $.package["rust-version"]     "1.85"
1  path        $.dependencies.tokio.version  "1.48"
2  value       $.dependencies.serde          "1.0"


`find_structured_path` parses every file under a directory in the configured format, so point it at files of that format:


In [ ]:
reports = TemporaryDirectory()
reports_root = Path(reports.name)
for day, latency in [("monday", 240), ("tuesday", 1800), ("wednesday", 2600)]:
    (reports_root / f"{day}.json").write_text(
        json.dumps({"day": day, "p99_ms": latency})
    )

slow_days = Searcher(
    "$.p99_ms > 1000",
    SearchConfig(structured_mode=StructuredMode.JSON, path_order=PathOrder.PATH),
)
for result in slow_days.find_structured_path(reports_root):
    print(
        Path(result.path).name,
        [(found.path, found.node_text) for found in result.matches],
    )

tuesday.json [('$.p99_ms', '1800')]
wednesday.json [('$.p99_ms', '2600')]


## Reading structured values

> **Warning:** `node_text` is always a `str` holding the value **exactly as the document writes it**: strings keep their quotes and escapes, numbers their spelling, such as `1_000.5` or `0xff`, and `null` is the text `"null"`. Used as they are, these strings give wrong answers without raising any error.

`value()` reads `node_text` into the Python value the document means, parsing it in Rust on each call. Objects, mappings, and tables become `dict`s, and arrays and sequences become `list`s. Read every value with it before computing with it, comparing it, or passing it on:


In [ ]:
def show_values(searcher: Searcher, document: str) -> None:
    """Prints each structured match's path and text with the value `value()` reads."""
    for found in searcher.find_structured(document):
        value = found.value()
        print(
            f"{found.path:14} {found.node_text!r:24} -> {value!r} ({type(value).__name__})"
        )


settings_json = (
    '{"replicas": 3, "latency_ms": 912, "budget_ms": 1000, '
    '"owner": "e\\u0041", "enabled": false, "region": null}'
)
settings = Searcher(
    ["$.replicas", "$.latency_ms", "$.budget_ms", "$.owner", "$.enabled", "$.region"],
    SearchConfig(structured_mode=StructuredMode.JSON),
)
show_values(settings, settings_json)

raw = {found.path: found.node_text for found in settings.find_structured(settings_json)}
print("\n912 > 1000 as text:", raw["$.latency_ms"] > raw["$.budget_ms"])

$.replicas     '3'                      -> 3 (int)
$.latency_ms   '912'                    -> 912 (int)
$.budget_ms    '1000'                   -> 1000 (int)
$.owner        '"e\\u0041"'             -> 'eA' (str)
$.enabled      'false'                  -> False (bool)
$.region       'null'                   -> None (NoneType)

912 > 1000 as text: True


TOML and YAML spell numbers, strings, booleans, and dates in their own ways, which `value()` resolves. TOML datetimes become `datetime`, `date`, or `time`.


In [13]:
service_toml = """\
name = "worker"
queue_limit = 1_000_000
mask = 0xff
started = 2026-03-18T10:15:00Z
path = 'C:\\jobs'
"""
service_yaml = """\
enabled: yes
owner: ~
mode: 0755
python: 3.10
timeout: 1:30
resources:
  cpu: 500m
  memory: 1Gi
"""
show_values(
    Searcher(
        ["$.name", "$.queue_limit", "$.mask", "$.started", "$.path"],
        SearchConfig(structured_mode=StructuredMode.TOML),
    ),
    service_toml,
)
print()
show_values(
    Searcher(
        ["$.enabled", "$.owner", "$.mode", "$.python", "$.timeout", "$.resources"],
        SearchConfig(structured_mode=StructuredMode.YAML),
    ),
    service_yaml,
)

$.name         '"worker"'               -> 'worker' (str)
$.queue_limit  '1_000_000'              -> 1000000 (int)
$.mask         '0xff'                   -> 255 (int)
$.started      '2026-03-18T10:15:00Z'   -> datetime.datetime(2026, 3, 18, 10, 15, tzinfo=datetime.timezone.utc) (datetime)
$.path         "'C:\\jobs'"             -> 'C:\\jobs' (str)

$.enabled      'yes'                    -> True (bool)
$.owner        '~'                      -> None (NoneType)
$.mode         '0755'                   -> 755 (int)
$.python       '3.10'                   -> 3.1 (float)
$.timeout      '1:30'                   -> 90 (int)
$.resources    'cpu: 500m\n  memory: 1Gi\n' -> {'cpu': '500m', 'memory': '1Gi'} (dict)


A TOML table comes back as a `dict`, including the subtables that later headers add, such as `[server.tls]` below. Its `node_text` can't always be read alone, since it spans every line the table's headers define, here `[logging]` too, so `value()` reads a TOML table from the searched text by its path.


In [ ]:
deploy_toml = """\
[server]
host = "0.0.0.0"
port = 8443

[logging]
level = "info"

[server.tls]
cert = "/etc/tls/cert.pem"

[[jobs]]
name = "build"
retries = 2

[[jobs]]
name = "deploy"
[jobs.step]
run = "make deploy"
"""
tables = Searcher(
    ["$.server", "$.jobs[1]"], SearchConfig(structured_mode=StructuredMode.TOML)
)
for found in tables.find_structured(deploy_toml):
    print(f"{found.path}: {found.value()}")

$.server: {'host': '0.0.0.0', 'port': 8443, 'tls': {'cert': '/etc/tls/cert.pem'}}
$.jobs[1]: {'name': 'deploy', 'step': {'run': 'make deploy'}}


Predicates need no reading: byteseek compares numbers, strings, booleans, and RFC 3339 timestamps itself. A predicate skips a value of another type, such as the number written as a string `"1800"` below.

> **Warning:** that skip is silent. Use `structured_predicate_handling=StructuredPredicateHandling.ERROR` to raise `SearchError` instead when a document might hold mistyped values:


In [ ]:
mixed_json = json.dumps(
    [{"latency_ms": 912}, {"latency_ms": "1800"}, {"latency_ms": 2400}]
)
for handling in StructuredPredicateHandling:
    predicate = Searcher(
        "$.latency_ms > 500",
        SearchConfig(
            structured_mode=StructuredMode.JSON, structured_predicate_handling=handling
        ),
    )
    print(f"{handling}:")
    try:
        show(predicate.find_structured(mixed_json))
    except ValueError as error:
        print(f"{type(error).__name__}: {error}")

ignore:
0  predicate   $[0].latency_ms  912
0  predicate   $[2].latency_ms  2400
error:
SearchError: structured predicate at pattern index 0 failed at $[1].latency_ms: path does not contain a numeric value


## Errors

`SearchError` is a `ValueError`. Every search method raises it for invalid patterns or options and for a structured text or file that doesn't parse, so a malformed document never passes for one without matches. A path that doesn't exist raises `FileNotFoundError`, as `open()` does.


In [17]:
attempts: list[Callable[[], object]] = [
    lambda: Searcher([]),
    lambda: events.count('{"events": ['),
    lambda: SearchConfig(max_matches=-1),
    lambda: SearchConfig(structured_mode="xml"),
    lambda: Searcher("ERROR").find_path("no/such/dir"),
]
for attempt in attempts:
    try:
        attempt()
    except (ValueError, OSError) as error:
        print(f"{type(error).__name__}: {error}")

SearchError: at least one pattern is required
SearchError: failed to parse structured input as Json: EOF while parsing a list at line 1 column 12
ValueError: max_matches must be at least 0, not -1
ValueError: 'xml' is not a valid StructuredMode
FileNotFoundError: [Errno 2] No such file or directory: 'no/such/dir'


## Searching from threads and processes

Searches release the GIL, so a thread pool searches several texts at once. Searchers and configs also pickle, compiling their patterns again when unpickled, so they can be sent to a process pool.


In [ ]:
texts = ["INFO ok\nERROR status=500 message=boom\n" * 400_000 for _ in range(8)]
errors = Searcher("error", SearchConfig(case_mode=CaseMode.ASCII_INSENSITIVE))


def timed(search: Callable[[], list[int]]) -> tuple[list[int], float]:
    """Runs `search` and returns its result with the time it took in milliseconds."""
    started = perf_counter()
    result = search()
    return result, (perf_counter() - started) * 1_000


sequential, sequential_ms = timed(lambda: [errors.count(text) for text in texts])
with ThreadPoolExecutor() as pool:
    threaded, threaded_ms = timed(lambda: list(pool.map(errors.count, texts)))

assert sequential == threaded
print(
    f"{sum(sequential):,} matches in {len(texts)} texts of {len(texts[0]) / 1e6:.0f} MB"
)
print(f"sequential: {sequential_ms:.1f} ms, threads: {threaded_ms:.1f} ms")

restored = pickle.loads(pickle.dumps(errors))
print(restored, restored.count(texts[0]) == sequential[0])

3,200,000 matches in 8 texts of 15 MB
sequential: 6.0 ms, threads: 4.0 ms
Searcher(['error'], SearchConfig(case_mode=CaseMode.ASCII_INSENSITIVE)) True


In [19]:
workspace.cleanup()
reports.cleanup()